In [ ]:
# Notebooks live in notebooks/; make the repo root importable (ppo, train_ppo, ...).
import sys
from pathlib import Path

_root = Path.cwd()
while not (_root / "ppo" / "paths.py").exists() and _root != _root.parent:
    _root = _root.parent
sys.path.insert(0, str(_root))


# `ppo_experiment` — usage walkthrough

This notebook demonstrates the modularized harness. The X-Y-Z code is:

- **X** — `num_fbs` (1 or 2)
- **Y** — scenario (1 = 1 MBS / 2000×1500, 2 = 2 MBSs / 4000×3000)
- **Z** — config (1 = reward gamma 0.0, 2 = reward gamma 0.1)

Tunable hooks (independent of the code): `ent_coef`, `action_scale`, `learning_rate`, `total_timesteps`, `max_episode_steps`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ppo_experiment import (
    ExperimentConfig,
    train,
    test,
    list_runs,
    list_test_results,
    load_test_result,
    SCENARIOS,
    CONFIGS,
    TRAINING_LOG,
    TEST_LOG_DIR,
)

## 1. Inspect the preset tables

These are the only two presets per axis; tweak the dicts in `ppo_experiment.py` to add more.

In [ ]:
print("SCENARIOS (Y):")
for k, v in SCENARIOS.items():
    print(f"  {k}: {v}")

print("\nCONFIGS (Z):")
for k, v in CONFIGS.items():
    print(f"  {k}: {v}")

## 2. List existing runs

Older runs (created before this script) won't have a `code` filled in — they predate `experiment_config.json`. New runs created via `train(...)` will.

In [ ]:
runs_df = list_runs()
runs_df.tail(15)

## 3. Train

`ExperimentConfig.from_code("X-Y-Z", ...)` builds the config; hook overrides go in as kwargs.

Switching the same training between the four common combinations is just changing the code string:

| code | what it means |
|------|---------------|
| `1-1-1` | 1 FBS, 1 MBS, gamma 0.0 |
| `1-1-2` | 1 FBS, 1 MBS, gamma 0.1 |
| `2-2-1` | 2 FBSs, 2 MBSs, gamma 0.0 |
| `2-2-2` | 2 FBSs, 2 MBSs, gamma 0.1 |

Each call to `train(...)`:
1. Saves the model into the next `ppo_runs/run_NNN/`
2. Writes `experiment_config.json` alongside it
3. Copies that run's `monitor.csv` into the run dir
4. Appends a row to `training_log.csv` (cumulative ledger)

In [ ]:
# Example A: 2 FBS, 2 MBSs, gamma=0.1
exp = ExperimentConfig.from_code(
    "2-2-2",
    total_timesteps=3_000,
    max_episode_steps=30,
    ent_coef=0.7,
    action_scale=0.9,
    learning_rate=1e-4,
)
print("code:", exp.code, "| gamma:", exp.gamma, "| scenario:", exp.scenario)

run_dir = train(exp)
print("saved ->", run_dir)

In [ ]:
# Example B (alternative): 1 FBS, 1 MBS, gamma=0.0 — just change the code
# exp = ExperimentConfig.from_code("1-1-1", total_timesteps=3_000, ent_coef=0.5, action_scale=0.6)
# train(exp)

## 4. Inspect the cumulative training ledger

Every `train(...)` call appends one row here, regardless of code or hook values.

In [ ]:
if TRAINING_LOG.exists():
    log = pd.read_csv(TRAINING_LOG)
    log.tail(10)
else:
    print("No training_log.csv yet — train at least once.")

## 5. Test the latest run

`test(run="latest")` loads the newest `run_NNN/` and rolls out the policy for `max_episode_steps`. The reset state is randomized unless you pass `custom_state`. Length of `custom_state` must be `5 * num_fbs`.

In [ ]:
# 2-FBS reset state: each block is [x, y, height, power, power_status]
custom_state = np.array([800.0, 800.0, 100.0, 10.5, 1.0], dtype=np.float32) 
# custom_state = np.array(
#     [
#         [0.0, 0.0, 100.0, 7.0, 0.0],
#         [2000.0, 300.0, 80.0, 8.5, 0.0],
#     ],
#     dtype=np.float32,
# )

state_df, metrics_df = test(
    run="run_039",
    custom_state=custom_state,
    max_episode_steps=100,
    action_scale_override=0.6,  
)

In [ ]:
state_df.head()

In [ ]:
metrics_df[["reward", "total_connected", "fbs_connected", "total_power"]].describe()

## 6. Test a specific run by name

In [ ]:
# 1-FBS reset (length = 5 * num_fbs = 5)
custom_state_1fbs = np.array([800.0, 800.0, 100.0, 10.5, 1.0], dtype=np.float32)
custom_state_2fbs = np.array(
    [
        [0.0, 0.0, 100.0, 7.0, 0.0],
        [2000.0, 300.0, 80.0, 8.5, 0.0],
    ],
    dtype=np.float32,
)

# Re-test run_039 — note: every test cell writes to the same `state_df` /
# `metrics_df` variable names so section 9 keeps working regardless of which
# test path you ran last.
state_df, metrics_df = test(
    run="run_039",
    custom_state=custom_state_1fbs,
    max_episode_steps=100,
)

## 7. Test a legacy run (no `experiment_config.json`)

Older runs were trained before this harness existed. To replay one, tell `test(...)` what code they correspond to via `exp_override`. The override is used only to rebuild the env with the matching world dimensions and MBS layout — the model weights are still loaded from the run dir.

In [ ]:
# Suppose run_039 was a 1-FBS / 1-MBS / gamma=0 run:
custom_state_1fbs = np.array([800.0, 800.0, 100.0, 10.5, 1.0], dtype=np.float32)
custom_state_2fbs = np.array(
    [
        [0.0, 0.0, 100.0, 7.0, 0.0],
        [2000.0, 300.0, 80.0, 8.5, 0.0],
    ],
    dtype=np.float32,
)

state_df_legacy, metrics_df_legacy = test(
    run="run_045",
    exp_override=ExperimentConfig.from_code("2-1-1"),
    custom_state=custom_state_2fbs,
    max_episode_steps=100,
)

## 8. Where the trajectories went

`test(...)` writes three files to `test_logs/<X-Y-Z>/<run>_<timestamp>_*`:

- `*_trajectory.csv` — the per-step state DataFrame
- `*_metrics.csv` — reward + info per step (with `user_positions` stripped, since it's an array per step)
- `*_meta.json` — the code, run dir, custom_state, full `ExperimentConfig`

So every test rollout is filed under the X-Y-Z folder it belongs to.

In [ ]:
for code_dir in sorted(TEST_LOG_DIR.glob("*")):
    files = sorted(code_dir.glob("*"))
    print(f"{code_dir.name}: {len(files)} files")
    for f in files[-6:]:
        print("  ", f.name)

## 9. Per-metric IEEE-style plots

Each metric gets its own figure so you can drop any single one into a draft. Styling reuses the IEEE rcParams defined in `plot_trajectories.py` (serif typography, hidden top/right spines, dotted grid, inward ticks). Default `figsize=(3.6, 2.6)` is sized for IEEE single-column width — pass a larger tuple for a double-column layout.

**Source toggle:** flip `USE_DISK` to `False` to plot the in-memory `metrics_df` from a prior `test(...)` call this session. With `USE_DISK=True` the cell reads back any saved rollout from `test_logs/<X-Y-Z>/`.

**Saving for the draft:** every panel function takes a `save=` path (PDF for vector, PNG at the IEEE_RC `savefig.dpi=400`). Each cell shows the suggested filename in a commented line.

In [ ]:
# --- pick your source --------------------------------------------------
USE_DISK = True   # flip to False to plot the in-memory metrics_df from sections 5-7

if USE_DISK:
    print(list_test_results().to_string(index=False))
    state_df, metrics_df, meta = load_test_result(
        code="1-1-1",          # X-Y-Z folder; set to None to search all codes
        run_name=None,         # e.g. "run_039" to pin a specific training run
        timestamp=None,        # e.g. "20260501_122954" for an exact rollout
        latest=True,
    )
    print(f"\nloaded: code={meta['code']} run={meta['run_dir']} ts={meta['timestamp']} "
          f"steps={meta['num_steps']}")
else:
    meta = {"code": "?", "run_dir": "?", "timestamp": "?"}

# Stem used for suggested save filenames in the per-metric cells below.
fig_stem = f"{meta['code']}_{meta['run_dir']}_{meta['timestamp']}"

In [ ]:
# IEEE-style single-panel metric helper — reused by the four cells below.
import matplotlib as mpl
from pathlib import Path
from plot_trajectories import IEEE_RC


def plot_metric(
    series,
    ylabel,
    *,
    figsize=(3.6, 2.6),
    xlabel="Step",
    color="#1f4e79",
    linewidth=1.4,
    marker="o",
    markersize=3.0,
    markevery=None,
    title=None,
    save=None,
):
    """Render a single time-series metric in IEEE journal style.

    Parameters
    ----------
    series : pandas Series indexed by step
    ylabel : axis label including units, e.g. 'Transmit power (W)'
    figsize : default (3.6, 2.6) — IEEE single-column width
    color : line + marker color (defaults to deep blue from IEEE_PALETTE)
    marker : marker glyph; pass `None` to drop markers entirely.
    markersize : in points (default 3.0 — small).
    markevery : draw a marker every N points; None = every step. Use this
        to thin markers when the series is dense.
    title : optional; usually omitted in print and described in the caption
    save : path (.pdf for vector, .png for raster). Saves at IEEE_RC dpi.
    """
    with mpl.rc_context(IEEE_RC):
        fig, ax = plt.subplots(figsize=figsize)
        ax.plot(
            series.index, series.values,
            color=color,
            linewidth=linewidth,
            marker=marker,
            markersize=markersize,
            markevery=markevery,
            markerfacecolor=color,
            markeredgecolor=color,
            markeredgewidth=0.5,
            solid_capstyle="round",
        )
        ax.set_xlabel(xlabel)
        ax.set_ylabel(ylabel)
        if title:
            ax.set_title(title, pad=4)
        ax.grid(True, linestyle=":", alpha=0.55)
        for spine in ("top", "right"):
            ax.spines[spine].set_visible(False)
        ax.tick_params(direction="in", length=3, width=0.7,
                       top=False, right=False)
        ax.margins(x=0.02)
        fig.tight_layout()
        if save is not None:
            Path(save).parent.mkdir(parents=True, exist_ok=True)
            fig.savefig(save, bbox_inches="tight")
    return fig, ax

In [ ]:
# Reward per step
plot_metric(
    metrics_df["reward"],
    ylabel="Reward",
    marker="^",markersize=3,markevery=3,
    # save=f"figures/reward_{fig_stem}.pdf",
)
plt.show()

In [ ]:
# Connected users per step
plot_metric(
    metrics_df["total_connected"],
    ylabel="Connected users",
    color="#2e7d32",   # forest green
    # save=f"figures/connected_{fig_stem}.pdf",
)
plt.show()

In [ ]:
# Total transmit power per step
plot_metric(
    metrics_df["total_power"],
    ylabel="Total transmit power (W)",
    color="#c0392b",   # crimson
    # save=f"figures/power_{fig_stem}.pdf",
)
plt.show()

In [ ]:
# Average per-user rate per step
plot_metric(
    metrics_df["avg_rate"],
    ylabel="Average rate (bps/Hz)",
    color="#6a1b9a",   # violet
    # save=f"figures/avg_rate_{fig_stem}.pdf",
)
plt.show()

In [ ]:
# Peek at the loaded trajectory: per-step state for every FBS
state_df.head()

In [ ]:
# 2D paths per FBS
fig, ax = plt.subplots(figsize=(5, 4))
fbs_ids = sorted({lvl0 for (lvl0, _) in state_df_legacy.columns if lvl0.startswith("fbs")})
for fbs in fbs_ids:
    ax.plot(state_df_legacy[(fbs, "x")], state_df_legacy[(fbs, "y")], marker=".", label=fbs)
ax.set_xlabel("x (m)")
ax.set_ylabel("y (m)")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 10. CLI usage (no notebook needed)

```bash
# Train
python ppo_experiment.py train --code 2-2-2 --total-timesteps 3000 --ent-coef 0.7 --action-scale 0.9

# Test the latest run
python ppo_experiment.py test --run latest --max-episode-steps 100

# Test a specific run with a custom reset state (5 numbers per FBS)
python ppo_experiment.py test --run run_039 --state "800,800,100,10.5,1"

# Test a legacy run by telling the harness what code it should use
python ppo_experiment.py test --run run_039 --code 1-1-1 --state "800,800,100,10.5,1"

# Browse runs
python ppo_experiment.py list
```